# Xi Codec v0 — Jupyter + SymPy Attack Notebook

This notebook attacks the reversible transforms and reproduces the compression/round-trip measurements.

In [1]:
from pathlib import Path
import csv, hashlib, importlib.util, json, sys, zlib
import sympy as sp

base = Path("/mnt/data")
spec = importlib.util.spec_from_file_location("xic", base / "xi_codec_v0.py")
xic = importlib.util.module_from_spec(spec)
sys.modules["xic"] = xic
spec.loader.exec_module(xic)

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

print("loaded", base / "xi_codec_v0.py")

loaded /mnt/data/xi_codec_v0.py


## 1. Assembly trace agreement

The Python port used by the codec is compared against the compiled assembly runner CSV for the first 192 rows.

In [2]:
s = xic.XiState()
fields = ["step","live_word","lowbit","carry_event","A","theta_ticks","kappa","u","v","uv","floor_den","window_ready","r_num","r_den","cL_num","cR_num","c_den"]
rows_checked = 0
with open(base / "xi_full_engine_trace_patched.csv") as f:
    for rows_checked, row in enumerate(csv.DictReader(f), 1):
        xic.xi_step(s)
        for name in fields:
            got = getattr(s, name)
            want = int(row[name])
            assert got == want, (rows_checked, name, got, want)
print({"assembly_rows_checked": rows_checked, "match": True})

{'assembly_rows_checked': 192, 'match': True}


## 2. SymPy byte-transform attacks

The codec uses two reversible residual transforms: XOR and subtraction modulo 256. SymPy/brute-force verifies all byte pairs.

In [3]:
x, p = sp.symbols("x p", integer=True)
# Full finite-domain attack over all byte values.
xor_ok = all(((a ^ b) ^ b) == a for a in range(256) for b in range(256))
sub_ok = all((((a - b) % 256 + b) % 256) == a for a in range(256) for b in range(256))
print({"xor_all_65536_pairs": xor_ok, "sub_mod256_all_65536_pairs": sub_ok})

{'xor_all_65536_pairs': True, 'sub_mod256_all_65536_pairs': True}


## 3. Round-trip and compression measurements

In [4]:
results = json.loads((base / "xi_compression_report.json").read_text())
for r in results:
    original = (base / r["file"]).read_bytes()
    restored = (base / (r["file"] + ".restored")).read_bytes()
    assert sha256_bytes(original) == sha256_bytes(restored)
    print({
        "file": r["file"],
        "raw": r["original_size"],
        "zlib9": r["zlib9_size"],
        "xic": r["encoded_size"],
        "raw_over_xic": round(r["ratio_original_over_encoded"], 6),
        "xi_minus_zlib": r["xi_vs_zlib9_delta_bytes"],
        "modes": {k:v for k,v in r["modes"].items() if v},
        "roundtrip": True,
    })

{'file': 'mock_text_repeated.txt', 'raw': 417272, 'zlib9': 12461, 'xic': 28088, 'raw_over_xic': 14.855882, 'xi_minus_zlib': 15627, 'modes': {'raw_zlib': 102}, 'roundtrip': True}
{'file': 'mock_xi_native.bin', 'raw': 262144, 'zlib9': 34230, 'xic': 388, 'raw_over_xic': 675.628866, 'xi_minus_zlib': -33842, 'modes': {'xi_direct_no_payload': 64}, 'roundtrip': True}
{'file': 'mock_xi_sparse_residual.bin', 'raw': 262144, 'zlib9': 34727, 'xic': 6282, 'raw_over_xic': 41.729386, 'xi_minus_zlib': -28445, 'modes': {'xi_xor_residual_zlib': 56, 'xi_sub_residual_zlib': 8}, 'roundtrip': True}
{'file': 'Unified_Quintic.pdf', 'raw': 691276, 'zlib9': 597455, 'xic': 622300, 'raw_over_xic': 1.11084, 'xi_minus_zlib': 24845, 'modes': {'raw_zlib': 169}, 'roundtrip': True}
{'file': 'lean4_theorem_proving.pdf', 'raw': 7449778, 'zlib9': 2641439, 'xic': 2755704, 'raw_over_xic': 2.703403, 'xi_minus_zlib': 114265, 'modes': {'raw_zlib': 1819}, 'roundtrip': True}


## 4. Local conclusion

The Xi-projection codec is reversible. It strongly compresses Xi-native or Xi-plus-sparse-residual data. On ordinary PDFs in this test, it falls back to raw zlib blocks and is larger than plain zlib because the current projector does not yet model PDF byte structure.